# The Kelly criterion and risk aversion

The previous notebook (`a_risky_gamble.ipynb`) showed that a positive-arithmetic-EV gamble can wipe out almost every player when wealth compounds. The fix is to bet only a fraction $f$ of wealth each round, chosen to maximize the **expected log growth rate**
$$g(f) \;=\; E\bigl[\log\bigl(1 + f\,(R - 1)\bigr)\bigr].$$
This is the **Kelly criterion** (Kelly 1956; see also Thorp 2011, *The Kelly Capital Growth Investment Criterion*). Below we (1) compute Kelly fractions for the doubling/quartering gamble, (2) solve a Thorp-style hot/cold blackjack variant with and without a minimum-bet constraint, and (3) show how a CRRA agent's risk aversion modifies the optimal fraction.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar, minimize

## 1. The Kelly fraction

For the doubling/quartering gamble of `a_risky_gamble.ipynb` ($R = 2$ or $0.25$, each w.p. $\tfrac{1}{2}$), what fraction $f$ of wealth should be bet each round?

In [2]:
def expected_log_growth_rate(f, p, x1, x2):
    """E[log(1 + f(R-1))] for a binary asset with payoffs x1, x2 and P(R=x1)=p."""
    return p * np.log(1 + f * (x1 - 1)) + (1 - p) * np.log(1 + f * (x2 - 1))

def find_optimal_fraction(p, x1, x2):
    result = minimize_scalar(lambda f: -expected_log_growth_rate(f, p, x1, x2),
                             bounds=(0, 1), method="bounded")
    return result.x

In [3]:
p, x1, x2 = 0.5, 2.0, 0.25

fractions = [0.02, 0.15, 0.5, 1.0]
rows = [(f, expected_log_growth_rate(f, p, x1, x2)) for f in fractions]
table = pd.DataFrame(rows, columns=["Fraction f", "E[log growth]"])
print(table.to_string(index=False, float_format="{:.4f}".format))

 Fraction f  E[log growth]
     0.0200         0.0023
     0.1500         0.0102
     0.5000        -0.0323
     1.0000        -0.3466


In [4]:
f_star = find_optimal_fraction(p, x1, x2)
print(f"Optimal fraction f*           = {f_star:.4f}")
print(f"Optimal expected log growth   = {expected_log_growth_rate(f_star, p, x1, x2):.4f}")

Optimal fraction f*           = 0.1667
Optimal expected log growth   = 0.0103


$f^* = 1/6 \approx 0.1667$: the player should bet one-sixth of wealth each round, not all of it. The arithmetic expected return ($1.125$) and the Kelly-optimal expected log-growth ($\approx 0.0103$) are pointing at very different objectives.

## 2. Hot and cold decks

A blackjack card-counter sees the deck state before each hand. When the deck is **hot** the win probability is $p_{\text{hot}} = 0.52$; when it is **cold** it is $p_{\text{cold}} = 0.48$. Each hand pays $2\times$ (win) or $0$ (lose). The two cases:

1. **Unconstrained.** Choose $f_{\text{hot}}$ and $f_{\text{cold}}$ independently. The bettor opts out of unfavorable hands.
2. **Constrained.** A house minimum-bet rule forces $f_{\text{hot}} = 3\,f_{\text{cold}}$ — the bettor must keep playing cold hands at one-third the hot-hand stake.

Following Thorp (2011), §1–2.

In [5]:
p_hot, p_cold = 0.52, 0.48
x_win, x_lose = 2.0, 0.0

In [6]:
def find_hot_cold_unconstrained(p_hot, p_cold, x_win, x_lose):
    return (find_optimal_fraction(p_hot,  x_win, x_lose),
            find_optimal_fraction(p_cold, x_win, x_lose))

def find_hot_cold_constrained(p_hot, p_cold, x_win, x_lose, ratio=3.0):
    """Maximize 0.5 g(f_hot) + 0.5 g(f_cold) subject to f_hot = ratio * f_cold."""
    def neg_expected_growth(f):
        f_hot, f_cold = f
        return -(0.5 * expected_log_growth_rate(f_hot,  p_hot,  x_win, x_lose) +
                 0.5 * expected_log_growth_rate(f_cold, p_cold, x_win, x_lose))
    result = minimize(neg_expected_growth,
                      x0=[0.1, 0.1], bounds=[(0, 1), (0, 1)],
                      constraints={"type": "eq",
                                   "fun": lambda f: f[0] - ratio * f[1]})
    return tuple(result.x)

In [7]:
f_unc = find_hot_cold_unconstrained(p_hot, p_cold, x_win, x_lose)
f_con = find_hot_cold_constrained  (p_hot, p_cold, x_win, x_lose, ratio=3.0)

def overall_growth(f_hot, f_cold):
    return 0.5 * expected_log_growth_rate(f_hot,  p_hot,  x_win, x_lose) + \
           0.5 * expected_log_growth_rate(f_cold, p_cold, x_win, x_lose)

summary = pd.DataFrame({
    "f_hot (%)":         [100 * f_unc[0], 100 * f_con[0]],
    "f_cold (%)":        [100 * f_unc[1], 100 * f_con[1]],
    "E[log growth] (%)": [100 * overall_growth(*f_unc),
                          100 * overall_growth(*f_con)],
}, index=["Unconstrained", "Constrained (f_hot = 3 f_cold)"])
print(summary.to_string(float_format="{:.2f}".format))

                                f_hot (%)  f_cold (%)  E[log growth] (%)
Unconstrained                        4.00        0.00               0.04
Constrained (f_hot = 3 f_cold)       2.41        0.80               0.02


Unconstrained, the bettor sits out cold hands ($f_{\text{cold}} = 0$) and bets $4\%$ on hot ones. The minimum-bet constraint forces participation at unfavorable odds and roughly halves the long-run growth rate.

## 3. How risk aversion changes the optimal betting fraction

Kelly betting maximizes $E[\log W]$, which is CRRA utility with relative risk aversion $\gamma = 1$. A more risk-averse agent ($\gamma > 1$) bets less; a less risk-averse one ($0 < \gamma < 1$) bets more.

We parameterize $\gamma$ via a more interpretable quantity $K$: the multiple of wealth in the win state that makes the agent indifferent to a fair coin flip whose lose state wipes out everything. Solving the indifference condition gives
$$\gamma \;=\; 1 - \frac{1}{\log_2 K} \qquad\Longleftrightarrow\qquad K \;=\; 2^{\,1/(1-\gamma)}.$$
Two cases below: $\gamma = 0.5$ (so $K = 4$) and $\gamma = 0.75$, which we will call **SBF utility** (so $K = 2^{4} = 16$ — the agent demands a sixteen-fold upside to accept a coin-flip ruin gamble). Both lie *below* log utility on the risk-aversion scale. We compute optimal fractions on two gambles: the original $\{2, 0.25\}$ from §1, then a higher-variance $\{3, 0.1\}$.

In [8]:
gamma_kelly = 1.0
gamma_crra  = 0.5
gamma_sbf   = 0.75

K_from_gamma = lambda g: 2 ** (1 / (1 - g))
print(f"gamma = {gamma_crra}  =>  K = {K_from_gamma(gamma_crra):g}")
print(f"gamma = {gamma_sbf}  =>  K = {K_from_gamma(gamma_sbf):g}  (SBF)")

gamma = 0.5  =>  K = 4
gamma = 0.75  =>  K = 16  (SBF)


In [9]:
def crra_utility(w, gamma):
    """CRRA utility u(w) with RRA coefficient gamma >= 0; gamma=1 is log."""
    if gamma == 1:
        return np.log(w)
    return (w ** (1 - gamma) - 1) / (1 - gamma)

def find_optimal_fraction_crra(p, x1, x2, gamma, w0=1.0):
    """Argmax_f  E[u(w0 * (1 + f(R-1)))]  for binary R in {x1, x2}."""
    def neg_eu(f):
        w_win  = w0 * (1 + f * (x1 - 1))
        w_lose = w0 * (1 + f * (x2 - 1))
        return -(p * crra_utility(w_win, gamma) + (1 - p) * crra_utility(w_lose, gamma))
    return minimize_scalar(neg_eu, bounds=(0, 1), method="bounded").x

### Original gamble: $R = 2$ or $R = 0.25$ each w.p. $\tfrac{1}{2}$

In [10]:
p, x1, x2 = 0.5, 2.0, 0.25

f_kelly_a = find_optimal_fraction_crra(p, x1, x2, gamma_kelly)
f_sbf_a   = find_optimal_fraction_crra(p, x1, x2, gamma_sbf)
f_crra_a  = find_optimal_fraction_crra(p, x1, x2, gamma_crra)

comparison_a = pd.DataFrame({
    "gamma":         [gamma_kelly, gamma_sbf, gamma_crra],
    "K":             [np.nan, K_from_gamma(gamma_sbf), K_from_gamma(gamma_crra)],
    "f*":            [f_kelly_a, f_sbf_a, f_crra_a],
    "E[log growth]": [expected_log_growth_rate(f_kelly_a, p, x1, x2),
                      expected_log_growth_rate(f_sbf_a,   p, x1, x2),
                      expected_log_growth_rate(f_crra_a,  p, x1, x2)],
}, index=["Kelly (log utility)", "SBF (gamma=0.75)", "CRRA (gamma=0.5)"])
print(comparison_a.to_string(float_format="{:.4f}".format))

                     gamma       K     f*  E[log growth]
Kelly (log utility) 1.0000     NaN 0.1667         0.0103
SBF (gamma=0.75)    0.7500 16.0000 0.2226         0.0092
CRRA (gamma=0.5)    0.5000  4.0000 0.3333        -0.0000


### Higher-variance gamble: $R = 3$ or $R = 0.1$ each w.p. $\tfrac{1}{2}$

In [11]:
p, x1, x2 = 0.5, 3.0, 0.1

f_kelly = find_optimal_fraction_crra(p, x1, x2, gamma_kelly)
f_sbf   = find_optimal_fraction_crra(p, x1, x2, gamma_sbf)
f_crra  = find_optimal_fraction_crra(p, x1, x2, gamma_crra)

comparison = pd.DataFrame({
    "gamma":         [gamma_kelly, gamma_sbf, gamma_crra],
    "K":             [np.nan, K_from_gamma(gamma_sbf), K_from_gamma(gamma_crra)],
    "f*":            [f_kelly, f_sbf, f_crra],
    "E[log growth]": [expected_log_growth_rate(f_kelly, p, x1, x2),
                      expected_log_growth_rate(f_sbf,   p, x1, x2),
                      expected_log_growth_rate(f_crra,  p, x1, x2)],
}, index=["Kelly (log utility)", "SBF (gamma=0.75)", "CRRA (gamma=0.5)"])
print(comparison.to_string(float_format="{:.4f}".format))

                     gamma       K     f*  E[log growth]
Kelly (log utility) 1.0000     NaN 0.3056         0.0777
SBF (gamma=0.75)    0.7500 16.0000 0.4121         0.0688
CRRA (gamma=0.5)    0.5000  4.0000 0.6111         0.0000


In both gambles, lower $\gamma$ means a bigger bet. The SBF agent ($\gamma = 0.75$, $K = 16$) sits between Kelly and the $\gamma = 0.5$ agent in each case. On the original $\{2, 0.25\}$ gamble, Kelly bets $1/6 \approx 16.7\%$, SBF bets $\approx 22.3\%$, and the $\gamma = 0.5$ agent bets twice Kelly at $1/3$ — and her expected log growth is exactly zero. On the higher-variance $\{3, 0.1\}$ gamble the same pattern holds at larger absolute fractions: Kelly $\approx 31\%$, SBF $\approx 41\%$, $\gamma = 0.5$ at $\approx 61\%$ — again with zero log-growth at $\gamma = 0.5$. Less risk aversion than log utility leaves you breaking even in the long run despite a positive arithmetic-EV gamble.

**Takeaway.** The Kelly criterion picks $f$ to maximize expected log growth. Information (hot/cold deck states) lets you bet more on favorable hands and zero on unfavorable ones; binding constraints (minimum-bet rules) cut growth roughly in half. Risk aversion above log utility shrinks the bet; risk aversion below log utility *grows* the bet but at the cost of long-run growth — log utility is the unique level that maximizes wealth growth in compounded play.